# 📂 Ficheros de texto en Python
*(operaciones básicas — versión explicada paso a paso)*

---

En este notebook cubrimos las **operaciones fundamentales** con ficheros de texto:

1. Abrir un fichero con `open()`, cerrarlo con `close()`.
2. Leer todo el contenido con `read()`.
3. Escribir en un fichero con `write()`.
4. Leer línea a línea con `readline()`, `readlines()` o `splitlines()`.
5. Usar `with` para garantizar el cierre automático.
6. Mover el cursor con `seek()`.

Todos los ejemplos usan el fichero **`mensaje.txt`** que está en esta carpeta.

## 1. Abrir y cerrar un fichero

La función `open()` recibe la **ruta** y el **modo**:

* `"r"` — read (leer). El fichero **debe existir**, si no da error.
* `"w"` — write (escribir). **Borra el fichero si existe**, lo crea si no.
* `"a"` — append (añadir al final). Crea el fichero si no existe.

Al terminar, hay que llamar a `.close()`. Es una buena costumbre, aunque enseguida veremos una forma más segura.

In [ ]:
fichero = open('mensaje.txt', 'r')
print(f"Nombre:  {fichero.name}")
print(f"Modo:    {fichero.mode}")
print(f"Cerrado: {fichero.closed}")

fichero.close()
print(f"Tras close(), ¿cerrado? {fichero.closed}")

## 2. Leer TODO el contenido con `read()`

`read()` devuelve el contenido completo del fichero como **una única cadena**. Es cómodo para ficheros pequeños, pero **peligroso** para ficheros de varios gigabytes: cargaría todo en RAM.

In [ ]:
fichero = open('mensaje.txt', 'r')
contenido = fichero.read()
fichero.close()

print(f'Longitud: {len(contenido)} caracteres')
print('---')
print(contenido)

## 3. Escribir en un fichero con `write()`

Con modo `"w"` creamos un fichero (o borramos si existía) y le escribimos con `.write()`. El método `write()` **NO añade salto de línea automático**: debes ponerlo tú con `\n`.

> ⚠️ **Cuidado con el modo `"w"`**: si `mi_fichero.txt` existía con contenido importante, abrirlo en modo `"w"` lo **borra sin preguntar**. Piénsalo dos veces.

In [ ]:
fichero = open('saludo.txt', 'w')
fichero.write('Hola desde Python\n')
fichero.write('Este es un fichero nuevo\n')
fichero.close()

# Comprobación: leer lo escrito
with open('saludo.txt') as f:
    print(f.read())

## 4. Leer línea a línea

Hay **tres formas** de recorrer un fichero línea a línea. Cada una tiene su lugar.

### 4a) `readline()` — una sola línea cada vez

Lee **una** línea del fichero, avanza el cursor y devuelve la línea (con `\n` incluido). Cuando llega al final, devuelve `""` (cadena vacía) — **no da error**.

In [ ]:
fichero = open('mensaje.txt', 'r')

n = 1
while True:
    linea = fichero.readline()
    if linea == '':
        break                       # fin de fichero
    print(f'{n:2}: {linea.rstrip()}')  # rstrip quita el \n final
    n = n + 1

fichero.close()

¡Ja! Uso del `while True` (combinado casi siempre con una sentencia `break`), recuerda que para comenzar te he recomendado que evites está práctica tan "**Pythonic**". Veamos como se hace sin él:

In [ ]:
fichero = open('mensaje.txt', 'r')
n = 1
linea = fichero.readline()
while linea != '':
    print(f'{n:2}: {linea.rstrip()}')
    n = n + 1
    linea = fichero.readline()

### 4b) `readlines()` — todas las líneas en una lista

Devuelve una **lista** con todas las líneas del fichero. Cada elemento es una cadena que incluye el `\n` final. Muy cómodo si ya sabes que el fichero cabe en memoria.

In [ ]:
with open('mensaje.txt') as f:
    lineas = f.readlines()

print(f'Total de líneas: {len(lineas)}')
print(f'Primera: {lineas[0]!r}')
print(f'Última:  {lineas[-1]!r}')

# Como es una lista, puedes usar TODOS los métodos del Tema 6
for i, linea in enumerate(lineas, 1):
    print(f'  {i}: {linea.rstrip()}')

### 4c) `read()` + `splitlines()`

Lees todo con `read()` y divides con `splitlines()`. La diferencia con `readlines()` es que **`splitlines()` NO deja el `\n`** en cada elemento. Muy cómodo para procesar.

In [ ]:
with open('mensaje.txt') as f:
    contenido = f.read()

lineas_limpias = contenido.splitlines()
print(f'Total: {len(lineas_limpias)} líneas SIN \\n al final\n')

for i, linea in enumerate(lineas_limpias, 1):
    print(f'  {i}: {linea!r}')

## 5. La REGLA DE ORO: siempre `with`

Todo el código anterior es correcto, pero **frágil**: si entre el `open` y el `close` ocurre una excepción, el fichero queda abierto. La solución elegante es el bloque `with`:

```python
with open('archivo.txt') as f:
    contenido = f.read()
# Al llegar aquí, el fichero YA está cerrado (garantizado)
```

Aunque haya una excepción dentro del bloque, Python **garantiza** el cierre. A partir de ahora **usa siempre `with`** para abrir ficheros. Este patrón se llama *gestor de contexto*.

In [ ]:
with open('mensaje.txt') as f:
    primera = f.readline()
    print(f'Primera línea leída: {primera.rstrip()}')

# f ya está cerrado aquí fuera
print(f'\n¿Cerrado tras el with? {f.closed}')

## 6. El cursor y `seek()`

Cada fichero abierto tiene un **cursor**: la posición actual dentro del fichero. Cada operación de lectura (`read`, `readline`) hace avanzar el cursor. Cuando llegas al final, futuras lecturas devuelven cadena vacía.

Para **rebobinar** o saltar a otra posición, se usa `.seek(posición_en_bytes)`. Lo más común es `f.seek(0)` para volver al inicio.

In [ ]:
with open('mensaje.txt') as f:
    contenido1 = f.read()
    print(f'1ª lectura: {len(contenido1)} caracteres')
    
    # El cursor está AL FINAL, aquí no queda nada
    contenido2 = f.read()
    print(f'2ª lectura: {len(contenido2)} caracteres  ← vacío')
    
    # Rebobinamos y volvemos a leer
    f.seek(0)
    contenido3 = f.read()
    print(f'Tras seek(0):  {len(contenido3)} caracteres')
    
    # Podemos también saltar a un byte concreto
    f.seek(50)
    trozo = f.read(20)
    print(f'Bytes 50-70: {trozo!r}')

## 🎯 Resumen: los patrones que verás una y otra vez

### Leer todo
```python
with open('archivo.txt') as f:
    contenido = f.read()
```

### Leer línea a línea (lo más común)
```python
with open('archivo.txt') as f:
    for linea in f:                # iterar directamente
        print(linea.rstrip())
```

### Escribir un fichero nuevo
```python
with open('salida.txt', 'w') as f:
    f.write('Línea 1\n')
    f.write('Línea 2\n')
```

### Añadir al final
```python
with open('log.txt', 'a') as f:
    f.write('Evento nuevo\n')
```

## 🚀 Para reflexionar

* ¿Cómo modificarías el ejemplo para contar **cuántas palabras** hay en `mensaje.txt`? (Pista: `contenido.split()` te lo da al momento.)
* ¿Y para saber cuántas líneas contienen la palabra `matemática`? (Con `if 'matemátic' in linea:`.)
* Si el fichero no existe, `open()` lanza `FileNotFoundError`. ¿Cómo lo tratarías con `try/except` (Tema 4)?